# Curso de Big Data e IA · Módulo 1
## Bloque 3: Sistemas y arquitecturas Big Data (Spark en Databricks)

**Duración estimada:** 3 horas

**Caso práctico:** los **cambios de Wikipedia en tiempo real** (Wikimedia EventStreams). Es un flujo público y gratuito:
**no hay que descargar ni subir ningún fichero**; el propio notebook captura los datos.

**Qué haremos:**
1. Capturar unos minutos del flujo y cargarlo en un DataFrame de Spark.
2. Filtros, agregaciones, joins, funciones de ventana y SQL.
3. Cache, guardado y errores típicos de los jobs de Spark.
4. **Structured Streaming**: leer lo mismo como flujo y ver cómo la tabla crece con datos nuevos.

**Importante:** Databricks Free Edition usa cómputo *serverless*: no hay Spark UI ni API de RDD, y los flujos
solo admiten el disparador `availableNow` (procesa todo lo pendiente y termina).
Si tu espacio de trabajo no llega a Wikimedia, el docente puede compartir un fichero ya capturado
(súbelo a la carpeta `wiki_eventos` del volumen, sección 1, y salta la captura de la sección 3).

## 1. Configuración: dónde guardamos los datos

In [ ]:
# AJUSTA estos nombres si tu catálogo o esquema son otros (Catalog en el menú lateral)
CATALOGO = "workspace"
ESQUEMA  = "default"
VOLUMEN  = "wikimedia"

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUMEN}")
BASE = f"/Volumes/{CATALOGO}/{ESQUEMA}/{VOLUMEN}"
print("Carpeta de trabajo:", BASE)

## 2. ¿Llega el flujo de Wikimedia a nuestro espacio de trabajo?

Wikimedia publica los cambios de todos sus proyectos como un flujo continuo (formato *Server-Sent Events*, sobre HTTP).
No hace falta cuenta ni clave. Si esta celda da `200`, el dominio está permitido desde tu espacio de trabajo.

In [ ]:
import requests

URL = "https://stream.wikimedia.org/v2/stream/recentchange"
CABECERAS = {"User-Agent": "CursoBigData/1.0 (formacion; contacto: tu-correo)"}   # pon tu correo

r = requests.get(URL, headers=CABECERAS, stream=True, timeout=15)
print(r.status_code, r.headers.get("content-type"))
r.close()

## 3. Capturar el flujo en ficheros

Una función lee el flujo durante unos segundos y guarda los eventos en **un fichero JSON por captura**
dentro del volumen. Capturamos **75 segundos** para tener datos suficientes y varios minutos de actividad.

In [ ]:
import json, time, os

def capturar(segundos=20, max_eventos=5000):
    """Lee el flujo de Wikimedia y guarda un fichero JSON (una línea por evento)."""
    destino = f"{BASE}/wiki_eventos"
    os.makedirs(destino, exist_ok=True)
    ruta = f"{destino}/captura_{int(time.time())}.json"
    n, inicio = 0, time.time()
    with requests.get(URL, headers=CABECERAS, stream=True, timeout=(10, 30)) as r, open(ruta, "w") as f:
        for linea in r.iter_lines():
            if linea.startswith(b"data:"):
                f.write(linea[5:].decode("utf-8").strip() + "\n")
                n += 1
            if n >= max_eventos or time.time() - inicio > segundos:
                break
    print(f"{n} eventos guardados en {ruta}")

capturar(segundos=75)

In [ ]:
# Mira qué hay en la carpeta
display(dbutils.fs.ls(f"{BASE}/wiki_eventos"))

## 4. Carga: del fichero JSON a un DataFrame

Ahora lo leemos como un **DataFrame normal** (lectura por lotes, `read`). Spark lee el fichero repartido en particiones.
Con un esquema explícito evitamos que Spark tenga que adivinar los tipos leyendo todos los datos.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, LongType, BooleanType

esquema_wiki = StructType([
    StructField("wiki",        StringType()),
    StructField("type",        StringType()),
    StructField("title",       StringType()),
    StructField("user",        StringType()),
    StructField("bot",         BooleanType()),
    StructField("server_name", StringType()),
    StructField("timestamp",   LongType()),
])

cambios = (spark.read
           .schema(esquema_wiki)
           .json(f"{BASE}/wiki_eventos")
           .withColumn("instante", F.to_timestamp(F.from_unixtime("timestamp"))))

print("Cambios capturados:", cambios.count())   # acción: lanza el trabajo
cambios.printSchema()

In [ ]:
display(cambios.limit(10))

## 5. Sintaxis básica de Spark DataFrames

Es muy parecida a pandas y a SQL, pero cada operación se ejecuta de forma distribuida.
Spark es **perezoso**: las transformaciones no se ejecutan hasta que pedimos un resultado (`show`, `count`, `display`).

In [ ]:
# Seleccionar columnas y filtrar: ediciones de la Wikipedia en español
(cambios
 .filter((F.col("wiki") == "eswiki") & (F.col("type") == "edit"))
 .select("instante", "title", "user", "bot")
 .show(5, truncate=False))

In [ ]:
# Tipos de cambio más frecuentes
(cambios.groupBy("type")
 .count()
 .orderBy(F.desc("count"))
 .show())

### El plan de ejecución
`explain()` muestra el plan físico que Spark ha elegido. Busca `Filter`, `HashAggregate` y `Exchange`
(un `Exchange` es un *shuffle*: datos que viajan entre máquinas). En Free Edition no hay Spark UI; usa también el **perfil de consulta**.

In [ ]:
(cambios.filter(F.col("type") == "edit")
 .groupBy("wiki").count()
 .explain())

## 6. Agregaciones: ¿cuánta actividad es de bots?

Contamos los cambios de cada Wikipedia y qué porcentaje ha hecho un bot. Nos quedamos con las que tienen actividad suficiente.

**Pregunta:** ¿qué proyecto tiene el mayor porcentaje de cambios hechos por bots?

In [ ]:
(cambios.groupBy("wiki")
 .agg(F.count("*").alias("cambios"),
      F.sum(F.when(F.col("bot") == True, 1).otherwise(0)).alias("de_bots"))
 .withColumn("bots_%", F.round(F.col("de_bots") / F.col("cambios") * 100, 1))
 .filter(F.col("cambios") >= 20)
 .orderBy(F.desc("cambios"))
 .show(10, truncate=False))

**Respuesta:** la que salga en tu tabla (depende del momento de la captura).
**Conclusión:** un `groupBy` con `agg` es el equivalente distribuido de una tabla dinámica; Spark calcula conteos parciales en cada partición y los combina.

## 7. Join: unir con una tabla pequeña

Tenemos el código de cada proyecto (`enwiki`, `eswiki`...), pero queremos el **nombre del idioma**.
Creamos una tabla pequeña a mano y la unimos con `join`, igual que en SQL. Como es pequeña, la marcamos con `broadcast`:
Spark la envía entera a cada máquina y evita mover los datos grandes.

In [ ]:
idiomas = spark.createDataFrame([
    ("enwiki", "inglés"), ("eswiki", "español"), ("frwiki", "francés"), ("dewiki", "alemán"),
    ("ruwiki", "ruso"), ("itwiki", "italiano"), ("jawiki", "japonés"), ("zhwiki", "chino"),
    ("commonswiki", "Commons (imágenes)"), ("wikidatawiki", "Wikidata (datos)"),
], "wiki string, idioma string")

(cambios
 .join(F.broadcast(idiomas), on="wiki", how="left")
 .withColumn("idioma", F.coalesce("idioma", F.lit("otros")))
 .groupBy("idioma")
 .agg(F.count("*").alias("cambios"))
 .orderBy(F.desc("cambios"))
 .show(12, truncate=False))

**Pregunta:** ¿por qué usamos `left` y no `inner`?
**Respuesta:** con `inner` perderíamos los cambios de proyectos que no están en la tabla; con `left` se conservan y los marcamos como `otros`.
**Conclusión:** el join de Spark es el de SQL; la tabla pequeña se difunde con `broadcast` para no mover la grande.

## 8. Funciones de ventana: los tres usuarios más activos de cada proyecto

Una ventana permite calcular algo "dentro de cada grupo" sin perder las filas, por ejemplo un ranking por proyecto.
`partitionBy` define el grupo y `orderBy` el orden dentro de él. Excluimos a los bots.

In [ ]:
por_usuario = (cambios.filter(F.col("bot") == False)
               .groupBy("wiki", "user")
               .agg(F.count("*").alias("cambios")))

ventana = Window.partitionBy("wiki").orderBy(F.desc("cambios"))

(por_usuario
 .withColumn("puesto", F.row_number().over(ventana))
 .filter(F.col("puesto") <= 3)
 .filter(F.col("wiki").isin("enwiki", "eswiki", "frwiki"))
 .orderBy("wiki", "puesto")
 .show(truncate=False))

## 9. Lo mismo con SQL

Spark permite consultar un DataFrame con SQL. Es el puente con las bases de datos relacionales que vimos en el Bloque 1.

In [ ]:
cambios.createOrReplaceTempView("cambios")
idiomas.createOrReplaceTempView("idiomas")

In [ ]:
%sql
SELECT COALESCE(i.idioma, 'otros') AS idioma,
       COUNT(*) AS cambios,
       SUM(CASE WHEN c.bot THEN 1 ELSE 0 END) AS de_bots
FROM cambios c
LEFT JOIN idiomas i ON c.wiki = i.wiki
GROUP BY COALESCE(i.idioma, 'otros')
ORDER BY cambios DESC
LIMIT 10

## 10. Cache y guardado

Si vamos a reutilizar un DataFrame varias veces, `.cache()` lo guarda en memoria (en serverless solo funciona `df.cache()`;
`CACHE TABLE` y `cacheTable()` no). Para guardar un resultado en un formato propio de Big Data usamos **Delta** (o Parquet).

In [ ]:
ediciones = cambios.filter(F.col("type") == "edit").cache()
ediciones.count()   # materializa la cache

In [ ]:
resumen = (ediciones.groupBy("wiki")
           .agg(F.count("*").alias("ediciones"),
                F.countDistinct("user").alias("usuarios")))

resumen.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOGO}.{ESQUEMA}.resumen_ediciones_por_wiki")
display(spark.table(f"{CATALOGO}.{ESQUEMA}.resumen_ediciones_por_wiki").orderBy(F.desc("ediciones")).limit(10))

In [ ]:
# Comprobación: la tabla debe aparecer en el catálogo (Catalog > workspace > default > Tables, o SQL Editor)
tablas = [t.tableName for t in spark.sql(f"SHOW TABLES IN {CATALOGO}.{ESQUEMA}").collect()]
print("Tablas en", f"{CATALOGO}.{ESQUEMA}:", tablas)
assert "resumen_ediciones_por_wiki" in tablas, "La tabla no se ha creado: revisa CATALOGO y ESQUEMA en la sección 1"

### 10.1 La caché tiene un coste
`cache()` ocupa memoria que luego falta para ejecutar. Mira dónde la guarda Spark y libérala cuando ya no la uses.

In [ ]:
print("Nivel de almacenamiento:", ediciones.storageLevel)   # memoria y disco
ediciones.unpersist()                                       # libera la memoria

### 10.2 Formatos de guardado: Parquet, Avro y Delta
- **Parquet**: por columnas, con esquema y compresión (snappy por defecto). Es el estándar para analítica.
- **Avro**: por filas, con el esquema en el fichero. Se usa para mover registros entre sistemas (por ejemplo con Kafka).
- **Delta**: ficheros Parquet más un registro de transacciones (versiones, escrituras atómicas). Es el formato por defecto de las tablas en Databricks.

**Cuidado con los ficheros:** Spark escribe una *carpeta* con un fichero por partición. Mira qué sale.

In [ ]:
ruta = f"{BASE}/salida_parquet"
resumen.write.mode("overwrite").parquet(ruta)
display(dbutils.fs.ls(ruta))                     # _SUCCESS y uno o varios ficheros part-...

ruta_uno = f"{BASE}/salida_un_fichero"
resumen.coalesce(1).write.mode("overwrite").parquet(ruta_uno)   # un único fichero part
display(dbutils.fs.ls(ruta_uno))

ruta_wiki = f"{BASE}/salida_por_wiki"
cambios.write.mode("overwrite").partitionBy("wiki").parquet(ruta_wiki)   # una carpeta por wiki
display(dbutils.fs.ls(ruta_wiki))

In [ ]:
# Avro: es un módulo externo; si tu entorno no lo trae, lo verás en el mensaje
try:
    resumen.write.mode("overwrite").format("avro").save(f"{BASE}/salida_avro")
    display(spark.read.format("avro").load(f"{BASE}/salida_avro"))
except Exception as e:
    print("Avro no disponible en este entorno:", str(e)[:200])

In [ ]:
# Delta: cada escritura crea una versión de la tabla
display(spark.sql(f"DESCRIBE HISTORY {CATALOGO}.{ESQUEMA}.resumen_ediciones_por_wiki"))

**Pregunta:** si guardas un resultado con Spark y salen varios ficheros, ¿qué ha pasado?
**Respuesta:** había varias particiones y cada una escribió el suyo.
**Conclusión:** para analítica, Parquet o Delta; piensa en cuántos ficheros salen antes de escribir.

## 11. Structured Streaming: leer la carpeta como un flujo

Spark trata el flujo como **una tabla que no deja de crecer**. Tú escribes la consulta igual que con un DataFrame
normal; Spark la ejecuta en micro-lotes sobre lo que va llegando.

- `readStream` en lugar de `read`.
- Un **esquema** explícito (un flujo no puede inferirlo bien).
- Un **checkpoint** por consulta: Spark apunta qué ficheros ya procesó. Así, al relanzar, solo lee lo nuevo.
- `trigger(availableNow=True)`: procesa todo lo pendiente y termina (el único modo que admite serverless).

Definimos dos funciones que reutilizaremos en todos los casos: una que **lee** el flujo y otra que lo **lanza**.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, BooleanType

esquema_wiki = StructType([
    StructField("wiki",        StringType()),
    StructField("type",        StringType()),
    StructField("title",       StringType()),
    StructField("user",        StringType()),
    StructField("bot",         BooleanType()),
    StructField("server_name", StringType()),
    StructField("timestamp",   LongType()),
])

def leer_wiki():
    """Flujo de la carpeta de capturas, con una columna de fecha legible."""
    return (spark.readStream
            .schema(esquema_wiki)
            .json(f"{BASE}/wiki_eventos")
            .withColumn("instante", F.to_timestamp(F.from_unixtime("timestamp"))))

def lanzar(flujo, tabla, nombre, modo="append"):
    """Escribe el flujo en una tabla Delta. Cada consulta tiene su propio checkpoint."""
    consulta = (flujo.writeStream
                .outputMode(modo)
                .option("checkpointLocation", f"{BASE}/ck_{nombre}")
                .trigger(availableNow=True)
                .toTable(f"{CATALOGO}.{ESQUEMA}.{tabla}"))
    consulta.awaitTermination()
    print(f"{tabla}: {spark.table(f'{CATALOGO}.{ESQUEMA}.{tabla}').count()} filas")

### 11.1 Caso 1: copia limpia del flujo

El caso más básico y el más usado en la práctica: guardar lo que llega, con sus tipos, en una tabla.
Es el equivalente a la capa **bronce** de la arquitectura medallón.

In [ ]:
lanzar(leer_wiki(), "wiki_cambios", "copia")
display(spark.table(f"{CATALOGO}.{ESQUEMA}.wiki_cambios").groupBy("wiki").count().orderBy(F.desc("count")).limit(10))

### Prueba del checkpoint
1. Vuelve a ejecutar la celda de **captura** (sección 3): se crea un fichero nuevo.
2. Ejecuta otra vez la celda del caso 1.
3. Mira cuántas filas había antes y cuántas hay ahora.

**Pregunta:** ¿Spark ha vuelto a leer los eventos de la primera captura?
**Respuesta:** no. El checkpoint recuerda qué ficheros ya procesó y solo lee los nuevos.
**Conclusión:** el flujo es incremental: cada ejecución cuesta lo que llega de nuevo, no todo el histórico.

In [ ]:
capturar(segundos=20)
lanzar(leer_wiki(), "wiki_cambios", "copia")

### 11.2 Verlo crecer: datos nuevos ronda a ronda

Con cómputo serverless el flujo no se queda escuchando, pero podemos **simular el tiempo real** repitiendo un ciclo:
llegan datos nuevos (captura de unos segundos) → Spark procesa **solo lo nuevo** → miramos cómo han cambiado las tablas.

En cada ronda verás:
- cuántas **filas nuevas** han entrado y el **total** acumulado (la tabla crece),
- el **% de cambios hechos por bots** (un resultado con estado que se va actualizando),
- las **3 Wikipedias** con más actividad.

Dura unos 2-3 minutos (6 rondas). Mientras corre, la celda va escribiendo una línea por ronda.

In [ ]:
def demo_en_vivo(rondas=6, segundos=10):
    tabla_copia = f"{CATALOGO}.{ESQUEMA}.wiki_cambios"
    por_tipo = (leer_wiki().groupBy("type", "bot").agg(F.count("*").alias("cambios")))
    antes = spark.table(tabla_copia).count() if spark.catalog.tableExists(tabla_copia) else 0
    historial = []
    for ronda in range(1, rondas + 1):
        capturar(segundos=segundos)                          # 1) llegan datos nuevos
        lanzar(leer_wiki(), "wiki_cambios", "copia")         # 2) Spark lee solo los ficheros nuevos
        lanzar(por_tipo, "wiki_por_tipo", "por_tipo", modo="complete")

        copia = spark.table(tabla_copia)
        total = copia.count()
        nuevos = total - antes
        antes = total
        bots = copia.filter(F.col("bot") == True).count()
        pct_bots = round(100 * bots / total, 1) if total else 0.0
        top3 = [f"{r['wiki']} ({r['count']})" for r in copia.groupBy("wiki").count().orderBy(F.desc("count")).limit(3).collect()]
        historial.append((ronda, total, nuevos, pct_bots))

        print(f"--- Ronda {ronda} · {time.strftime('%H:%M:%S')} ---")
        print(f"  filas nuevas: {nuevos:>5} {'█' * min(60, nuevos // 20)}")
        print(f"  total tabla : {total:>5}")
        print(f"  % bots      : {pct_bots}%")
        print(f"  top 3 wikis : {', '.join(top3)}")
    return historial

historial = demo_en_vivo(rondas=6, segundos=10)

In [ ]:
# La evolución en una tabla (pulsa "+" en el resultado para crear un gráfico de líneas con total por ronda)
display(spark.createDataFrame(historial, "ronda int, total long, nuevos long, pct_bots double"))

**Pregunta:** ¿por qué cada ronda tarda lo mismo aunque la tabla ya sea más grande?
**Respuesta:** porque Spark solo procesa los ficheros nuevos; el checkpoint ya recuerda los anteriores.
**Conclusión:** así se ve un flujo: la tabla crece con cada llegada y los resultados con estado (% de bots) se actualizan sin recalcular el histórico.

### 11.3 Caso 2: filtrar el flujo

Una consulta **sin estado**: cada evento se mira por separado. Nos quedamos con las **ediciones hechas por personas**
(no bots) en las Wikipedias en inglés, español y francés.

In [ ]:
humanas = (leer_wiki()
           .filter((F.col("type") == "edit") & (F.col("bot") == False)
                   & F.col("wiki").isin("enwiki", "eswiki", "frwiki"))
           .select("instante", "wiki", "title", "user"))

lanzar(humanas, "wiki_ediciones_humanas", "humanas")
display(spark.table(f"{CATALOGO}.{ESQUEMA}.wiki_ediciones_humanas").orderBy(F.desc("instante")).limit(10))

### 11.4 Caso 3: contar por tipo de cambio y por bot

Ahora una consulta **con estado**: Spark tiene que recordar los conteos entre lotes.
Usamos `outputMode("complete")`: en cada lote se reescribe el resultado completo.

**Pregunta:** ¿qué proporción de los cambios los hacen bots?

In [ ]:
por_tipo = (leer_wiki()
            .groupBy("type", "bot")
            .agg(F.count("*").alias("cambios")))

lanzar(por_tipo, "wiki_por_tipo", "por_tipo", modo="complete")
display(spark.table(f"{CATALOGO}.{ESQUEMA}.wiki_por_tipo").orderBy(F.desc("cambios")))

**Respuesta:** la que salga en tu tabla (depende del momento de la captura).
**Conclusión:** un conteo en streaming es un `groupBy` normal; Spark guarda el estado y lo actualiza con cada lote.

### 11.5 Caso 4: ventanas de tiempo

En streaming casi siempre se pregunta *"¿cuántos por minuto?"*. Se hace con una **ventana** sobre la hora del evento
(`window`). Aquí contamos cambios por ventana de 1 minuto.

In [ ]:
por_minuto = (leer_wiki()
              .groupBy(F.window("instante", "1 minute"))
              .agg(F.count("*").alias("cambios")))

lanzar(por_minuto, "wiki_por_minuto", "por_minuto", modo="complete")
display(spark.table(f"{CATALOGO}.{ESQUEMA}.wiki_por_minuto")
        .select(F.col("window.start").alias("minuto"), "cambios")
        .orderBy("minuto"))

**Pregunta:** ¿hay minutos con muchos más cambios que otros?
**Respuesta:** míralo en la tabla: la actividad de Wikipedia varía durante el día y entre minutos.
**Conclusión:** las ventanas convierten un flujo de eventos sueltos en una serie temporal, lista para un gráfico o una alerta.

### 11.6 Varias consultas, una sola fuente

Hemos lanzado **cuatro consultas** sobre la misma carpeta. Cada una tiene su **propio checkpoint** (`ck_copia`,
`ck_humanas`, `ck_por_tipo`, `ck_por_minuto`), así que cada una avanza a su ritmo.

Prueba: ejecuta otra **captura** y relanza **solo una** de las consultas. Las demás tablas no cambian hasta que las relances.

**Conclusión:** una misma fuente puede alimentar muchas consultas independientes; el checkpoint es lo que las separa.

In [ ]:
# Comprobación: las consultas de streaming escriben tablas Delta en el catálogo
tablas = [t.tableName for t in spark.sql(f"SHOW TABLES IN {CATALOGO}.{ESQUEMA}").collect()]
esperadas = ["wiki_cambios", "wiki_ediciones_humanas", "wiki_por_tipo", "wiki_por_minuto"]
print("Faltan:", [t for t in esperadas if t not in tablas] or "ninguna")
# Desde el SQL Editor se consultan con el nombre completo: SELECT * FROM workspace.default.wiki_cambios

## 12. Errores típicos de los jobs de Spark

Con unos miles de filas nada de esto falla, y justo por eso conviene verlo: **el código es el mismo con cientos de millones de filas**.
Usamos el DataFrame `cambios` de la sección 4.

### 12.1 `collect()` trae todo al driver
`collect()` y `toPandas()` mueven *todos* los datos a una sola máquina. Aquí cabe; con datos grandes el driver se queda sin memoria.

**Pregunta:** ¿quieres ver 20 filas, qué usas? **Respuesta:** `limit(20).show()`.

In [ ]:
# PELIGROSO con datos grandes (aquí son pocos miles de filas, así que funciona):
filas = cambios.collect()
print("Filas traídas al driver:", len(filas))
del filas

# SEGURO: calcula en el clúster y trae solo el resultado
cambios.limit(20).show(5)
por_wiki = cambios.groupBy("wiki").count().collect()   # pocas filas: esto sí es razonable
print("Proyectos distintos:", len(por_wiki))

### 12.2 Variables que no se comparten
Una variable de Python modificada dentro de una función distribuida no se actualiza en el driver: cada executor trabaja con su copia.
Para sumar o contar, **agrega**.

In [ ]:
# Forma correcta: que cuente el clúster
n_bots = cambios.agg(F.sum(F.when(F.col("bot") == True, 1).otherwise(0))).first()[0]
print("Cambios hechos por bots (calculado en el clúster):", n_bots)

### 12.3 Desbalanceo (skew): una clave domina
Mira cómo se reparten los cambios por proyecto. Si un proyecto concentra una parte enorme, un `groupBy("wiki")` envía todas
esas filas a una sola partición, y una tarea tarda mucho más que las demás.

In [ ]:
por_wiki = cambios.groupBy("wiki").count().orderBy(F.desc("count"))
por_wiki.show(5)

total  = cambios.count()
maximo = por_wiki.first()["count"]
print(f"El proyecto con más cambios concentra el {maximo/total*100:.1f} % del total")

### 12.4 Particiones: ni pocas ni demasiadas
El número de particiones decide cuántas tareas hay. Con pocos datos y 200 particiones de shuffle, casi todas quedan vacías
(AQE las une automáticamente). En serverless este parámetro se gestiona solo.

In [ ]:
# Muestra el valor configurado (puede ser "auto" en Databricks)
print("shuffle.partitions =", spark.conf.get("spark.sql.shuffle.partitions"))

### 12.5 Trampas en el código
- **UDF de Python** frente a funciones nativas: usa `F.lower`, `F.when`... siempre que puedas.
- **Ventanas sin `partitionBy`**: la ventana abarca todo el conjunto (compara con la sección 8, que sí lleva `partitionBy("wiki")`).
- **Broadcast de tablas grandes**: solo difunde las pequeñas (umbral por defecto: 10 MB).

In [ ]:
# Función nativa en lugar de una UDF: título en minúsculas
cambios.select("title", F.lower("title").alias("titulo_min")).show(5, truncate=False)

### 12.6 Recalcular sin cache
Cada acción vuelve a leer y procesar los ficheros desde el principio. Compara el tiempo de la misma consulta
sobre el DataFrame sin cache y sobre el cacheado de la sección 10.

In [ ]:
import time
def tiempo(f):
    t = time.time(); f(); return round(time.time() - t, 2)

sin_cache = lambda: cambios.filter(F.col("type") == "edit").groupBy("wiki").count().collect()
con_cache = lambda: ediciones.groupBy("wiki").count().collect()
print("Sin cache:", tiempo(sin_cache), "s")
print("Con cache:", tiempo(con_cache), "s")

### 12.7 Muchos ficheros pequeños
Cada captura crea un fichero nuevo. Con miles de ficheros pequeños, leer es lento: Spark paga un coste por abrir cada uno.

In [ ]:
ficheros = dbutils.fs.ls(f"{BASE}/wiki_eventos")
print("Ficheros en la carpeta:", len(ficheros))
print("Tamaño medio (bytes):", round(sum(f.size for f in ficheros) / max(len(ficheros), 1)))

## 13. Verlo en vivo, evento a evento

Hasta ahora hemos trabajado con capturas. Aquí ves el flujo **tal como llega**. En **Databricks Free (serverless) no se puede dejar
una consulta de Structured Streaming ejecutándose de forma continua** (solo `availableNow`), así que lo hacemos de dos maneras:
1. Leer el flujo directamente y escribir cada evento según llega (sin Spark).
2. Repetir *captura + lectura con Spark* en rondas cortas y ver crecer los contadores (microlotes hechos a mano).

In [ ]:
# 13.1 Cada evento, línea a línea, durante 30 segundos
inicio, n = time.time(), 0
with requests.get(URL, headers=CABECERAS, stream=True, timeout=(10, 30)) as r:
    for linea in r.iter_lines():
        if not linea.startswith(b"data:"):
            continue
        try:
            e = json.loads(linea[5:].decode("utf-8"))
        except ValueError:
            continue
        n += 1
        print(f"{time.strftime('%H:%M:%S')}  {e.get('wiki','?'):<12} {e.get('type','?'):<10} {str(e.get('user','?'))[:18]:<18} {str(e.get('title',''))[:50]}", flush=True)
        if time.time() - inicio > 30:
            break
print(f"--- {n} eventos en 30 s ---")

In [ ]:
# 13.2 Contadores que crecen: cada ronda captura 10 s y vuelve a leer todo con Spark
from IPython.display import clear_output
carpeta = f"{BASE}/wiki_vivo"
os.makedirs(carpeta, exist_ok=True)
for ronda in range(1, 7):
    ruta = f"{carpeta}/captura_{int(time.time())}.json"
    inicio = time.time()
    with requests.get(URL, headers=CABECERAS, stream=True, timeout=(10, 30)) as r, open(ruta, "w") as f:
        for linea in r.iter_lines():
            if linea.startswith(b"data:"):
                f.write(linea[5:].decode("utf-8").strip() + "\n")
            if time.time() - inicio > 10:
                break
    vivo = spark.read.schema(esquema_wiki).json(carpeta)
    clear_output(wait=True)
    print(f"Ronda {ronda}/6 · {time.strftime('%H:%M:%S')} · eventos acumulados: {vivo.count()}")
    vivo.groupBy("wiki").count().orderBy(F.desc("count")).show(5)

**Pregunta:** ¿esto es streaming de verdad?
**Respuesta:** la 13.1 sí lee el flujo en directo, pero sin Spark. La 13.2 son microlotes que lanzamos nosotros; Structured Streaming hace lo mismo solo, con un checkpoint, cuando el cluster lo permite.
**Conclusión:** en Free usamos `availableNow` y bucles; en un cluster normal, `trigger(processingTime=...)` mantiene la consulta viva.

## Ejercicio propuesto
1. Calcula el porcentaje de cambios hechos por bots en cada proyecto y muestra los 5 con mayor porcentaje (con al menos 20 cambios).
2. ¿Cuántos cambios hay de cada tipo (`edit`, `new`, `log`, `categorize`...) en cada ventana de 1 minuto? Usa la sección 11.5 como modelo.
3. Compara el tiempo de una agregación con y sin `.cache()`.
4. Haz una **nueva captura** y relanza el caso 1 de streaming: ¿cuántas filas nuevas entran? ¿Por qué no se vuelven a leer las anteriores?

## Limpieza (opcional)

In [ ]:
import shutil
for t in ("wiki_cambios", "wiki_ediciones_humanas", "wiki_por_tipo", "wiki_por_minuto", "resumen_ediciones_por_wiki"):
    spark.sql(f"DROP TABLE IF EXISTS {CATALOGO}.{ESQUEMA}.{t}")
for c in ("ck_copia", "ck_humanas", "ck_por_tipo", "ck_por_minuto", "wiki_eventos", "salida_parquet", "salida_un_fichero", "salida_por_wiki", "salida_avro", "wiki_vivo"):
    shutil.rmtree(f"{BASE}/{c}", ignore_errors=True)